In [13]:
import pandas as pd
import numpy as np

import boto3
import logging 

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder

In [6]:
logger = logging.getLogger()
logger.setLevel(logging.INFO)
logger.addHandler(logging.StreamHandler())

In [9]:
logger.debug("Starting preprocessing.")

# S3 parameters
bucket_name = "sagemaker-project-p-yapqplufpbzj"
object_key = "churn-prediction-mlops-p-yapqplufpbzj/data/ecommerce_customer_churn_dataset.csv"
download_path = "tests/local_file.csv"  # where the file will be saved locally

# Create an S3 client
s3 = boto3.client("s3")

# Download the file
s3.download_file(bucket_name, object_key, download_path)

print(f"File downloaded from s3://{bucket_name}/{object_key} to {download_path}")


File downloaded from s3://sagemaker-project-p-yapqplufpbzj/churn-prediction-mlops-p-yapqplufpbzj/data/ecommerce_customer_churn_dataset.csv to tests/local_file.csv


In [19]:
logger.debug("Reading downloaded data.")
df = pd.read_csv(
    download_path,
    # header=0,
    # names=feature_columns_names + [label_column],
    # dtype=merge_two_dicts(feature_columns_dtype, label_column_dtype),
)

In [20]:
df

,Age,Gender,Country,City,Membership_Years,Login_Frequency,Session_Duration_Avg,Pages_Per_Session,Cart_Abandonment_Rate,Wishlist_Items,...,Email_Open_Rate,Customer_Service_Calls,Product_Reviews_Written,Social_Media_Engagement_Score,Mobile_App_Usage,Payment_Method_Diversity,Lifetime_Value,Credit_Balance,Churned,Signup_Quarter
0,43.0,Male,France,Marseille,2.9,14.0,27.4,6.0,50.6,3.0,...,17.9,9.0,4.0,16.3,20.8,1.0,953.33,2278.0,0,Q1
1,36.0,Male,UK,Manchester,1.6,15.0,42.7,10.3,37.7,1.0,...,42.8,7.0,3.0,NaN,23.3,3.0,1067.47,3028.0,0,Q4
2,45.0,Female,Canada,Vancouver,2.9,10.0,24.8,1.6,70.9,1.0,...,0.0,4.0,1.0,NaN,8.8,NaN,1289.75,2317.0,0,Q4
3,56.0,Female,USA,New York,2.6,10.0,38.4,14.8,41.7,9.0,...,41.4,2.0,5.0,85.9,31.0,3.0,2340.92,2674.0,0,Q1
4,35.0,Male,India,Delhi,3.1,29.0,51.4,NaN,19.1,9.0,...,37.9,1.0,11.0,83.0,50.4,4.0,3041.29,5354.0,0,Q4
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
49995,38.0,Female,USA,Los Angeles,10.0,10.0,35.8,8.8,51.2,3.0,...,9.2,5.0,1.0,20.7,25.2,1.0,2365.98,3499.0,0,Q2
49996,37.0,Male,USA,Chicago,1.4,21.0,32.4,9.6,32.5,2.0,...,17.6,5.0,3.0,38.6,12.1,4.0,1293.60,3154.0,0,Q3
49997,44.0,Female,USA,Phoenix,2.8,19.0,40.3,10.4,50.6,1.0,...,25.2,4.0,6.0,59.9,NaN,1.0,1655.28,1622.0,0,Q1
49998,41.0,Female,USA,Chicago,2.9,17.0,29.9,10.1,60.8,1.0,...,34.6,5.0,2.0,NaN,14.1,3.0,1286.11,2634.0,0,Q2


In [21]:
feature_columns_names = [
    # Customer Demographics
    "Age",
    "Gender",
    "Country",
    "City",
    "Membership_Years",

    # Platform Engagement
    "Login_Frequency",
    "Session_Duration_Avg",
    "Pages_Per_Session",
    "Cart_Abandonment_Rate",
    "Wishlist_Items",
    "Email_Open_Rate",
    "Mobile_App_Usage",
    "Social_Media_Engagement_Score",

    # Purchase Behavior
    "Total_Purchases",
    "Average_Order_Value",
    "Days_Since_Last_Purchase",
    "Discount_Usage_Rate",
    "Returns_Rate",
    "Payment_Method_Diversity",

    # Customer Service & Value
    "Customer_Service_Calls",
    "Product_Reviews_Written",
    "Lifetime_Value",

    # Financial / Temporal
    "Credit_Balance",
    "Signup_Quarter",
]

label_column = "Churned"

feature_columns_dtype = {
    # Customer Demographics
    "Age": np.float64,
    "Gender": str,
    "Country": str,
    "City": str,
    "Membership_Years": np.float64,

    # Platform Engagement
    "Login_Frequency": np.float64,
    "Session_Duration_Avg": np.float64,
    "Pages_Per_Session": np.float64,
    "Cart_Abandonment_Rate": np.float64,
    "Wishlist_Items": np.float64,
    "Email_Open_Rate": np.float64,
    "Mobile_App_Usage": np.float64,
    "Social_Media_Engagement_Score": np.float64,

    # Purchase Behavior
    "Total_Purchases": np.float64,
    "Average_Order_Value": np.float64,
    "Days_Since_Last_Purchase": np.float64,
    "Discount_Usage_Rate": np.float64,
    "Returns_Rate": np.float64,
    "Payment_Method_Diversity": np.float64,

    # Customer Service & Value
    "Customer_Service_Calls": np.float64,
    "Product_Reviews_Written": np.float64,
    "Lifetime_Value": np.float64,

    # Financial / Temporal
    "Credit_Balance": np.float64,
    "Signup_Quarter": str,
}

label_column_dtype = {"Churned": np.float64}

In [23]:
logger.info("Defining transformers.")
categorical_features = ["Gender", "Country", "City", "Signup_Quarter"]
numeric_features = [elt for elt in feature_columns_names if elt not in categorical_features]

numeric_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ]
)

categorical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="constant", fill_value="missing")),
        ("onehot", OneHotEncoder(handle_unknown="ignore")),
    ]
)

preprocess = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features),
    ]
)

Defining transformers.


In [25]:
logger.info("Applying transforms.")
y = df.pop("Churned")

Applying transforms.


In [26]:
X_pre = preprocess.fit_transform(df)


In [27]:
y_pre = y.to_numpy().reshape(len(y), 1)

X = np.concatenate((y_pre, X_pre), axis=1)

logger.info("Splitting %d rows of data into train, validation, test datasets.", len(X))
np.random.shuffle(X)
train, validation, test = np.split(X, [int(0.7 * len(X)), int(0.85 * len(X))])

# logger.info("Writing out datasets to %s.", base_dir)
# pd.DataFrame(train).to_csv(f"{base_dir}/train/train.csv", header=False, index=False)
# pd.DataFrame(validation).to_csv(
#     f"{base_dir}/validation/validation.csv", header=False, index=False
# )
# pd.DataFrame(test).to_csv(f"{base_dir}/test/test.csv", header=False, index=False)

Splitting 50000 rows of data into train, validation, test datasets.


In [29]:
train.shape

(35000, 76)

In [36]:
np.unique(train[:,0], return_counts=True)

(array([0., 1.]), array([24908, 10092]))

In [49]:
from sklearn.linear_model import LogisticRegression

from sklearn.ensemble import GradientBoostingClassifier

In [50]:
clf = GradientBoostingClassifier(n_estimators=1000, max_depth=7)

In [51]:
clf.fit(train[:, 1:], train[:,0])

,loss,'log_loss'
,learning_rate,0.1
,n_estimators,1000
,subsample,1.0
,criterion,'friedman_mse'
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_depth,7
,min_impurity_decrease,0.0
,init,None


In [52]:
preds_train = clf.predict(train[:, 1:])
preds_val = clf.predict(validation[:, 1:])
preds_test = clf.predict(test[:, 1:])

In [53]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, classification_report

report_train = classification_report(train[:,0], preds_train)
print(report_train)

              precision    recall  f1-score   support

         0.0       1.00      1.00      1.00     24908
         1.0       1.00      1.00      1.00     10092

    accuracy                           1.00     35000
   macro avg       1.00      1.00      1.00     35000
weighted avg       1.00      1.00      1.00     35000



In [54]:
report_val = classification_report(validation[:,0], preds_val)
print(report_val)

              precision    recall  f1-score   support

         0.0       0.92      0.97      0.94      5306
         1.0       0.92      0.79      0.85      2194

    accuracy                           0.92      7500
   macro avg       0.92      0.88      0.90      7500
weighted avg       0.92      0.92      0.92      7500



In [55]:
report_test = classification_report(test[:,0], preds_test)
print(report_test)

              precision    recall  f1-score   support

         0.0       0.92      0.97      0.95      5336
         1.0       0.92      0.80      0.85      2164

    accuracy                           0.92      7500
   macro avg       0.92      0.88      0.90      7500
weighted avg       0.92      0.92      0.92      7500

